In [1]:
from fast_borf.weighted.borf_multi import BorfPipelineBuilder
from fast_borf.classes.bag_of_receptive_fields_sax.borf_multi import BorfPipelineBuilder as BorfPipelineBuilderOld
from aeon.datasets import load_classification
from fast_borf.pipeline.to_scipy import ToScipySparse
from fast_borf.pipeline.zero_columns_remover import ZeroColumnsRemover
from fast_borf.pipeline.reshaper import ReshapeTo2D
import xarray as xr

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
from sklearn.linear_model import RidgeClassifierCV

In [3]:
from irregular_ts.data_utils import data_new_folder
import xarray as xr
df = xr.open_dataset(data_new_folder() / "Garment.h5", engine="my_engine")["data"]
y, split = df.irr.get_task_target_and_split()
X, _ = df.irr.to_dense(
    concatenate_time=True,
    normalize_time=True,
)
train_idxs, test_idxs = split == "train", split == "test"
X_train, y_train = X[train_idxs], y[train_idxs]
X_test, y_test = X[test_idxs], y[test_idxs]

In [4]:
borf_builder = BorfPipelineBuilder(
    pipeline_objects=[
        (ReshapeTo2D, {}),
        (ZeroColumnsRemover, {}),
        (ToScipySparse, {})
    ],
    contains_time_idx=True
)
borf = borf_builder.build(
    X_train
)

In [5]:
# import numba
# numba.config.DISABLE_JIT = True

In [6]:
X_train[0, 0, :]

array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan])

In [7]:
borf.fit(X_train, y_train)
out1 = borf.transform(X_train)

In [8]:
out1

<18x14470 sparse matrix of type '<class 'numpy.int64'>'
	with 36370 stored elements in Compressed Sparse Row format>